# 06. Middleware and Guardrails

Learn the **middleware** system and **guardrails** used by LangChain v1 agents.


## Learning Objectives

- **Middleware concepts:** Understand how to add hooks to each stage of the agent execution pipeline
- **Built-in middleware:** Use built-in middleware such as `SummarizationMiddleware`
- **Custom middleware:** Implement custom middleware with `@before_model`, `@after_model`, `@wrap_model_call`, and `@dynamic_prompt`
- **Guardrails:** Learn how to block unsafe input and output


## 6.1 Environment Setup


In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
)

print("모델 준비 완료:", model.model_name)

In [ ]:
# Optional observability setup: LangSmith or Langfuse
# Set the keys in .env, or uncomment the lines below to enter them manually.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: automatically enabled when LANGSMITH_TRACING=true
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: pass config={"callbacks": [langfuse_handler]} to invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 6.2 Middleware Concepts

Middleware is the mechanism that **adds hooks to each stage of the agent execution pipeline** so you can control how the agent behaves.

![Middleware pipeline](../assets/images/middleware_pipeline.png)

**Five middleware hooks:**

| Hook | When it runs | Main Use Case |
|-----|----------|----------|
| `@before_model` | Before a model call | Input validation, message editing, guardrails |
| `@after_model` | After a model response | Output logging, response filtering |
| `@wrap_model_call` | Around a model call | Retry, fallback, caching |
| `@wrap_tool_call` | Around a tool call | Control tool execution |
| `@dynamic_prompt` | During prompt creation | Runtime prompt changes |


## 6.3 Built-In Middleware

LangChain v1 provides **built-in middleware** for common patterns. `SummarizationMiddleware` automatically summarizes earlier messages when a conversation becomes long, reducing token usage.


In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def search(query: str) -> str:
    """정보를 검색합니다."""
    return f"'{query}'에 대한 검색 결과"

# SummarizationMiddleware — 긴 대화를 자동 요약
from langchain.agents.middleware import SummarizationMiddleware

summarization = SummarizationMiddleware(
    model=model,
    trigger=("messages", 10),
)

agent_with_summary = create_agent(
    model=model,
    tools=[search],
    system_prompt="당신은 유용한 어시스턴트입니다.",
    middleware=[summarization],
)
print("SummarizationMiddleware 에이전트 생성 완료")

## 6.4 Custom Middleware: `@before_model`

The `@before_model` decorator runs **before the model is called**.

Common uses:
- Logging input messages
- Modifying or filtering messages
- Input validation (guardrails)
- Adding context


In [ ]:
from langchain.agents.middleware import before_model

@before_model
def log_model_input(state, runtime):
    """모델에 전송하기 전에 메시지를 로깅합니다."""
    msg_count = len(state["messages"])
    print(f"  모델 입력: {msg_count}개 메시지")

agent_logged = create_agent(
    model=model,
    tools=[search],
    system_prompt="당신은 유용한 어시스턴트입니다.",
    middleware=[log_model_input],
)

print("모델 호출 로깅 테스트:")
result = agent_logged.invoke(
    {"messages": [{"role": "user", "content": "Python 튜토리얼을 검색해 주세요"}]},
    config=lf_config,
)
print("응답:", result["messages"][-1].content[:200])

## 6.5 Custom Middleware: `@after_model`

The `@after_model` decorator runs **after the model response has been generated**.

Common uses:
- Logging model output
- Filtering or modifying responses
- Monitoring tool calls
- Validating output quality


In [ ]:
from langchain.agents.middleware import after_model

@after_model
def log_model_output(state, runtime):
    """모델 출력 생성 후 로깅합니다."""
    msg = state["messages"][-1] if state["messages"] else None
    if msg and hasattr(msg, 'content') and msg.content:
        print(f"  모델 출력: {msg.content[:100]}...")
    if msg and hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"  도구 호출: {[tc['name'] for tc in msg.tool_calls]}")

agent_full_log = create_agent(
    model=model,
    tools=[search],
    system_prompt="당신은 유용한 어시스턴트입니다.",
    middleware=[log_model_input, log_model_output],
)

print("전체 로깅 테스트:")
result = agent_full_log.invoke(
    {"messages": [{"role": "user", "content": "LangChain v1 기능을 검색해 주세요"}]},
    config=lf_config,
)

## 6.6 `@wrap_model_call`

The `@wrap_model_call` decorator **wraps the model call itself**, which lets you implement retry, fallback, caching, and similar patterns.

You execute the original model call through the `handler` function and can add custom logic before or after it.


In [ ]:
from langchain.agents.middleware import wrap_model_call
import time

@wrap_model_call
def retry_on_error(request, handler):
    """실패 시 지수 백오프로 모델 호출을 재시도합니다."""
    max_retries = 2
    for attempt in range(max_retries + 1):
        try:
            return handler(request)
        except Exception as e:
            if attempt < max_retries:
                wait = 2 ** attempt
                print(f"  재시도 {attempt + 1}/{max_retries} ({wait}초 대기)")
                time.sleep(wait)
            else:
                raise

agent_retry = create_agent(
    model=model,
    tools=[search],
    system_prompt="당신은 유용한 어시스턴트입니다.",
    middleware=[retry_on_error],
)
print("재시도 미들웨어 에이전트 생성 완료")

## 6.7 `@dynamic_prompt`

The `@dynamic_prompt` decorator **changes the system prompt dynamically at runtime**.

Common uses:
- Adding the current date and time
- Per-user prompt customization
- Changing behavior based on state
- A/B testing


In [ ]:
from langchain.agents.middleware import dynamic_prompt
from datetime import datetime

@dynamic_prompt
def add_datetime_context(request):
    """시스템 프롬프트에 현재 날짜와 시간을 추가합니다."""
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return f"현재 날짜와 시간: {now}\n\n당신은 유용한 어시스턴트입니다."

agent_dynamic = create_agent(
    model=model,
    tools=[search],
    middleware=[add_datetime_context],
)

result = agent_dynamic.invoke(
    {"messages": [{"role": "user", "content": "현재 날짜와 시간이 어떻게 되나요?"}]},
    config=lf_config,
)
print("동적 프롬프트 응답:", result["messages"][-1].content)

## 6.8 `@wrap_tool_call`

The `@wrap_tool_call` decorator **wraps a tool call itself**, so you can add custom logic before and after tool execution.

Like `@wrap_model_call`, it uses a `handler` function to run the original tool. You can use it for timing, logging, and error handling.

Common uses:
- **Measuring execution time:** monitor performance by tool
- **Logging:** record tool input and output
- **Error handling:** apply fallback behavior if a tool fails
- **Access control:** block or restrict specific tools


In [ ]:
from langchain.agents.middleware import wrap_tool_call
import time

@wrap_tool_call
def tool_timing_logger(request, handler):
    """실행 시간을 측정하고 도구 입출력을 로깅합니다."""
    tool_name = request.tool_call["name"]
    tool_args = request.tool_call["args"]
    print(f"  [도구 시작] {tool_name} | 입력: {tool_args}")

    start = time.perf_counter()
    try:
        result = handler(request)
        elapsed = time.perf_counter() - start
        print(f"  [도구 완료] {tool_name} | 소요 시간: {elapsed:.3f}s | 출력: {str(result)[:100]}")
        return result
    except Exception as e:
        elapsed = time.perf_counter() - start
        print(f"  [도구 실패] {tool_name} | 소요 시간: {elapsed:.3f}s | 에러: {e}")
        raise

agent_tool_logged = create_agent(
    model=model,
    tools=[search],
    system_prompt="당신은 유용한 어시스턴트입니다. 검색 도구를 사용하여 정보를 찾으세요.",
    middleware=[tool_timing_logger],
)

print("도구 호출 타이밍/로깅 미들웨어 테스트:")
result = agent_tool_logged.invoke(
    {"messages": [{"role": "user", "content": "LangChain 미들웨어 문서를 검색해 주세요"}]},
    config=lf_config,
)
print("\n최종 응답:", result["messages"][-1].content[:200])

## 6.9 Simple Guardrails

Middleware can also act as a lightweight guardrail. In the example below, a `before_model` hook blocks requests that contain prohibited keywords before the model is called.


In [ ]:
# 키워드 기반 가드레일
@before_model
def keyword_guardrail(state, runtime):
    """금지된 키워드가 포함된 요청을 차단합니다."""
    prohibited = ["hack", "exploit", "malware"]
    last_msg = state["messages"][-1]
    content = last_msg.content if hasattr(last_msg, 'content') else str(last_msg)
    
    for keyword in prohibited:
        if keyword.lower() in content.lower():
            raise ValueError(f"요청이 안전 정책에 의해 차단되었습니다.")

agent_guarded = create_agent(
    model=model,
    tools=[search],
    system_prompt="당신은 유용한 어시스턴트입니다.",
    middleware=[keyword_guardrail],
)

# 안전한 요청
result = agent_guarded.invoke(
    {"messages": [{"role": "user", "content": "Python 튜토리얼을 검색해 주세요"}]},
    config=lf_config,
)
print("안전한 요청:", result["messages"][-1].content[:100])

# 차단되는 요청
try:
    result = agent_guarded.invoke(
        {"messages": [{"role": "user", "content": "웹사이트 해킹 방법"}]}
    )
except ValueError as e:
    print(f"차단된 요청: {e}")

## 6.10 Summary

This notebook covered:

| Topic | Core API | Description |
|------|---------|-------------|
| Built-in middleware | `SummarizationMiddleware` | Automatically summarizes long conversations |
| Before-model hook | `@before_model` | Logs, validates, or modifies input before model execution |
| After-model hook | `@after_model` | Logs or validates model output after generation |
| Wrapped model call | `@wrap_model_call` | Adds retry, fallback, or caching around model calls |
| Dynamic prompt | `@dynamic_prompt` | Changes the system prompt at runtime |
| Wrapped tool call | `@wrap_tool_call` | Adds logging, timing, and control around tool execution |
| Guardrails | `@before_model` and middleware | Blocks unsafe or disallowed input |

### Next Steps
→ **[07_hitl_and_runtime.ipynb](./07_hitl_and_runtime.ipynb)**: Learn about human-in-the-loop, runtime context, and MCP.
